In [ ]:
import pandas as pd
import re
import spacy
import torch
import nltk
import numpy as np
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
from transformers import AutoModelForSequenceClassification
from nltk.sentiment.vader import SentimentIntensityAnalyzer
tqdm.pandas()

In [ ]:
file_path = "/content/cleaned_data_for_absa.csv"
unprocessed_data = pd.read_csv(file_path)

### Data Cleaning

In [ ]:
unprocessed_data.isna().sum()

,0
location,0
name,0
text,0


In [ ]:
unprocessed_data.shape

(2767, 3)

In [ ]:
unprocessed_data.head()

,location,name,text
0,Chicken Republic - Ilupeju,DR KHAN,Well maintened with taste
1,Chicken Republic - Ilupeju,Abuja Garden & Guest House,A place to patronising
2,Chicken Republic - Ilupeju,Olamide Adeyemi,I get my breakfast from Chicken Republic almos...
3,Chicken Republic - Ilupeju,Shanawaz A,Good
4,Chicken Republic - Ilupeju,Blessing Odunayo,The rice needs to be properly seasoned\nIt tas...


In [ ]:
data_to_drop = ['name']
data = unprocessed_data.drop(data_to_drop, axis=1)

In [ ]:
special_chars= r'[^a-zA-Z0-9 ]'
def process_texts(text):
  cleaned_text = re.sub(special_chars, '', text)
  return cleaned_text


In [ ]:
data['processed_review'] = data['text'].map(process_texts)

In [ ]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
data['processed_review'].head()

,processed_review
0,Well maintened with taste
1,A place to patronising
2,I get my breakfast from Chicken Republic almos...
3,Good
4,The rice needs to be properly seasonedIt taste...


### Sentence Decomposition

In [ ]:
nlp = spacy.load('en_core_web_lg')

In [ ]:
CONJ_PATTERN = r'\b(?:but|however|although|though|yet|while)\b'

def split_clauses(sentence):
    parts = re.split(CONJ_PATTERN, sentence, flags=re.IGNORECASE)
    return [p.strip() for p in parts if len(p.strip()) > 5]

def split_sentences(text):
    doc = nlp(text)
    return [sent.text.strip() for sent in doc.sents if sent.text.strip()]

def split_on_and(sentence):
    doc = nlp(sentence)
    splits = []
    current = []

    for token in doc:
        if token.text.lower() == "and":
            if any(t.pos_ == "VERB" for t in current):
                splits.append(" ".join(t.text for t in current))
                current = []
                continue
        current.append(token)

    if current:
        splits.append(" ".join(t.text for t in current))

    return [s.strip() for s in splits if len(s.strip()) > 5]

def decompose_review(text):
    sentences = split_sentences(text)
    clauses = []

    for sent in sentences:
        parts = split_clauses(sent)
        for part in parts:
            subparts = split_on_and(part)
            clauses.extend(subparts)
    if clauses:
      return clauses
    else:
      return text


text = "The food tasted good, but that water had a taste, which was terrible, and the manager was very rude ad disrespectful."
# text = "I am a man. She is a woman. I love apples."
decompose_review(text)

['The food tasted good ,',
 'that water had a taste , which was terrible ,',
 'the manager was very rude ad disrespectful .']

In [ ]:
data['simple_reviews'] = data['processed_review'].progress_apply(decompose_review)
data = data.explode('simple_reviews').reset_index(drop=True)

100%|██████████| 2767/2767 [01:15<00:00, 36.64it/s] 


### Clustering Texts

In [ ]:
def cluster_centroid(embeddings):
  return np.mean(embeddings, axis=0)


def similarity_to_centroid(embeddings, centroid):
  return cosine_similarity(embeddings, centroid.reshape(1, -1)).flatten()


In [ ]:
def filter_outliers(embeddings, z_thres=-1.0):
  centroid = cluster_centroid(embeddings)
  sims = similarity_to_centroid(embeddings, centroid)
  mean = sims.mean()
  std = sims.std() + 1e-8
  z_scores = (sims - mean) / (std)

  return z_scores >= z_thres

### Extracting Sentiment & Aspects

In [ ]:
nltk.download('vader_lexicon')
sid = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [ ]:
def extract_aspects(doc):
    ner_heads = {ent.root.idx: ent for ent in doc.ents}
    aspects = set()

    target_deps = {"nsubj", "dobj", "pobj"}

    for token in doc:
        for child in token.children:
            if child.dep_ in target_deps and not child.is_stop:
                aspects.add(
                    ner_heads[child.idx].text if child.idx in ner_heads else child.text
                )

    return list(aspects) if aspects else []


In [ ]:
get_sentiment = pipeline(
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    return_all_scores=True,
    truncation=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
aspect_n_modifiers = data['simple_reviews'].progress_apply(lambda txt: extract_aspects(nlp(txt)))

100%|██████████| 3439/3439 [00:20<00:00, 164.56it/s]


In [ ]:
records = []
for i, pair in aspect_n_modifiers.items():
  for aspect in pair:
    records.append({'id': i, 'aspect': aspect})

df_aspects = pd.DataFrame(records)

In [ ]:
df_aspects.head()

,id,aspect
0,0,taste
1,2,ambience
2,2,ChickWizzI
3,2,breakfast
4,2,Chicken Republic


In [ ]:
data = data.drop(columns=['review_idx'], errors='ignore')

data = data.reset_index(drop=True)
data['review_idx'] = data.index

data2 = data.merge(
    df_aspects.rename(columns={'id': 'review_idx'}),
    on='review_idx',
    how='left'
)


In [ ]:
data2['sentiment']  = data2['simple_reviews'].progress_apply(lambda txt: get_sentiment(txt)[0]['label'])

100%|██████████| 4849/4849 [00:52<00:00, 93.18it/s]


This uses dependecy trees a lot, so look well into it.

In [ ]:
txt = 'there was a hair in our food'
txt2 = 'I hate the tomatoes'
txt3 = 'Great fresh food prep One of the best McDonalds I ve been to Unruly school kids ruined the experience'
txt4 = 'Well when I got home I took the first bite on the sandwich'
docs2 = nlp(txt4)

for token in docs2:
  if token.dep_ == 'ROOT':
    print(f'{token} | {token.dep_} | {token.pos_}')
  for child in token.children:
    print(f'token: {child} | part of speech: {child.dep_}')

token: when | part of speech: advmod
token: I | part of speech: nsubj
token: home | part of speech: advmod
took | ROOT | VERB
token: Well | part of speech: intj
token: got | part of speech: advcl
token: I | part of speech: nsubj
token: bite | part of speech: dobj
token: the | part of speech: det
token: first | part of speech: amod
token: on | part of speech: prep
token: sandwich | part of speech: pobj
token: the | part of speech: det


In [ ]:
data2.isna().sum()

,0
location,0
text,0
processed_review,0
simple_reviews,0
review_idx,0
aspect,1758
sentiment,0


In [ ]:
data2['aspect'].fillna('No aspects detected', inplace=True)

/tmp/ipykernel_1370/3448171625.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data2['aspect'].fillna('No aspects detected', inplace=True)


In [ ]:
def pre_group_cleaning(txt):
    if not isinstance(txt, str):
        return None

    txt = txt.strip().lower()
    if not txt:
        return None

    if re.fullmatch(r'\d+', txt):
        return None

    if re.fullmatch(r"\d+\s*(minutes?|mins?|hours?|hrs?|secs?|seconds?)", txt):
        return None

    doc = nlp(txt)
    token = doc[0]

    if token.is_stop:
        return None

    return token.lemma_

In [ ]:
cleaned_aspects = data2['aspect'].progress_apply(lambda txt: pre_group_cleaning(txt))

100%|██████████| 4849/4849 [00:32<00:00, 147.39it/s]


In [ ]:
data2['aspect'] = cleaned_aspects
data2.dropna(inplace=True)

In [ ]:
aspects_w_n_na = data2[data2['aspect']!='No aspects detected']
aspect_series = aspects_w_n_na['aspect']
cleaned_aspects = [x for x in (pre_group_cleaning(txt) for txt in aspect_series) if x is not None]
unique_aspects = list(set(cleaned_aspects))

In [ ]:
print(len(cleaned_aspects))
print(len(unique_aspects))

3036
772


In [ ]:
embby_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Clustering

Before clustering, the embeddings for the topics must first be created.

In [ ]:
aspect_count = dict(Counter(data2['aspect']))

In [ ]:
filtered_aspects = [asp_k for asp_k, asp_v in aspect_count.items() if asp_v > 5]

In [ ]:
da_embeddings = embby_model.encode(filtered_aspects)

Now that the embeddings have been made, the next stage is to find the ideal number of clusters for agglomerative clustering

In [ ]:
cluster_range = [5, 10, 15, 20, 25, 30, 35]
best_score = -1
best_k = None

for k in cluster_range:
  clusterer = AgglomerativeClustering(n_clusters=k, metric='cosine', linkage='average')
  labels = clusterer.fit_predict(da_embeddings)

  # Calculate how clean the separations are
  score = silhouette_score(da_embeddings, labels, metric='cosine')
  print(f"Clusters: {k} | Silhouette Score: {score:.4f}")

  if score > best_score:
      best_score = score
      best_k = k

Clusters: 5 | Silhouette Score: 0.0411
Clusters: 10 | Silhouette Score: 0.0689
Clusters: 15 | Silhouette Score: 0.0995
Clusters: 20 | Silhouette Score: 0.1215
Clusters: 25 | Silhouette Score: 0.1168
Clusters: 30 | Silhouette Score: 0.1175
Clusters: 35 | Silhouette Score: 0.1177


The highest silehoutte score is when k = 20, so the ideal n_cluster will be 20:

In [ ]:
clustering = AgglomerativeClustering(n_clusters=20, metric='cosine', linkage='average')
labels = clustering.fit_predict(da_embeddings)

In [ ]:
word_clusters = pd.DataFrame({'Word': filtered_aspects, 'Cluster': labels})

In [ ]:
cluster_dict = {}

In [ ]:
for i in range(20):
  cluster_words = word_clusters[word_clusters['Cluster'] == i]['Word'].to_list()
  print(", ".join(cluster_words[:15]))

time, money, family, fly
traffic, car, space, environment, place, atmosphere, parking, area, location, park, road
server, security
taste, experience, quality, improvement, variety, response
staff, people, service, customer, attendant, person, worker, company
nigeria, republic, lagos
jollof, day
chicken, rice, snack, food, meal, cream, lunch, plate, pie, dish, fry, bite, eatery, restaurant, chip
order, pack, delivery, package
lot, combo, thing
chickwizz, jibowu
price, max, rate
ilupeju, moi
guy, friend, kid, nice, good, u
junction, outlet, branch, ac
ambience, relaxation
counter
menu
spot, point
meet


In [ ]:
clusters = defaultdict(list)

for _, row in word_clusters.iterrows():
  clusters[row['Cluster']].append(row['Word'])

In [ ]:
# Cluster count dictionary
cluster_val_count = defaultdict(list)

for clust_n, asps in clusters.items():
  asps_count = 0
  for asp in asps:
    asps_count += aspect_count[asp]
  cluster_val_count[clust_n].append(asps_count)

In [ ]:
cluster_reps = {}

for k, val in clusters.items():
    cluster_embeddings = embby_model.encode(np.array(val))
    centroid = cluster_centroid(cluster_embeddings)
    similarity_scores = similarity_to_centroid(cluster_embeddings, centroid)
    best_match_idx = np.argmax(similarity_scores)
    rep_topic = val[best_match_idx]
    cluster_reps[k] = rep_topic

In [ ]:
cluster_reps

{3: 'quality',
 15: 'relaxation',
 7: 'meal',
 6: 'jollof',
 0: 'money',
 4: 'customer',
 5: 'nigeria',
 9: 'lot',
 1: 'place',
 14: 'outlet',
 11: 'price',
 13: 'good',
 8: 'package',
 12: 'moi',
 17: 'menu',
 10: 'chickwizz',
 2: 'security',
 18: 'spot',
 16: 'counter',
 19: 'meet'}

In [ ]:
word_clusters['Topic'] = word_clusters['Cluster'].progress_map(lambda x: cluster_reps[x])

100%|██████████| 89/89 [00:00<00:00, 156911.75it/s]


In [ ]:
topic_words_dict = word_clusters.groupby('Topic')['Word'].apply(list).to_dict()

In [ ]:
word_clusters_dict = dict(zip(word_clusters.Word, word_clusters.Topic))

In [ ]:
data2['Topic'] = data2['aspect'].progress_map(lambda x: word_clusters_dict.get(x, np.nan))

100%|██████████| 3050/3050 [00:00<00:00, 406683.21it/s]


In [ ]:
data2.dropna(inplace=True)

There is a need to dissolve one of the groups into something finer.

In [ ]:
customer_words = topic_words_dict['customer']

for word in customer_words:
    word_clusters.loc[
        word_clusters['Word'] == word,
        'Topic'
    ] = word

In [ ]:
word_clusters_dict = dict(
    zip(word_clusters.Word, word_clusters.Topic)
)

data2['Topic'] = data2['aspect'].map(word_clusters_dict)

In [ ]:
data2['Topic'].replace({'worker': 'staff', 'person': 'people', 'jollof': 'meal'}, inplace=True)

/tmp/ipykernel_1370/156582829.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data2['Topic'].replace({'worker': 'staff', 'person': 'people', 'jollof': 'meal'}, inplace=True)


In [ ]:
invalid_topics = ['day', 'money', 'nigeria', 'lot', 'outlet', 'good', 'ilepeju', 'chickwizz', 'spot', 'counter', 'meet', 'moi', ]
data2 = data2[~data2['Topic'].isin(invalid_topics)]

In [ ]:
data2['Topic'].unique()

array(['quality', 'relaxation', 'meal', 'staff', 'people', 'service',
       'place', 'customer', 'price', 'package', 'menu', 'security',
       'attendant', 'company'], dtype=object)

In [ ]:
data2['Group_Label'] = data2.groupby(['Topic', 'sentiment']).ngroup()

In [ ]:
concat_review_df = data2.groupby(['Group_Label', 'Topic', 'sentiment'])['simple_reviews'].apply(
    lambda reviews: " ".join(reviews.dropna().astype(str))
).reset_index(name='concat_review')

In [ ]:
reviews_list = concat_review_df['concat_review'].to_list()

In [ ]:
data2 = data2.merge(concat_review_df[['Group_Label', 'concat_review']], on='Group_Label', how='left')

In [ ]:
data2.head(3)

,location,text,processed_review,simple_reviews,review_idx,aspect,sentiment,Topic,Group_Label,concat_review_x,concat_review_y
0,Chicken Republic - Ilupeju,Well maintened with taste,Well maintened with taste,Well maintened with taste,0,taste,positive,quality,29,Well maintened with taste The vibe was fun and...,Well maintened with taste The vibe was fun and...
1,Chicken Republic - Ilupeju,I get my breakfast from Chicken Republic almos...,I get my breakfast from Chicken Republic almos...,I get my breakfast from Chicken Republic almos...,2,ambience,positive,relaxation,32,I get my breakfast from Chicken Republic almos...,I get my breakfast from Chicken Republic almos...
2,Chicken Republic - Ilupeju,I get my breakfast from Chicken Republic almos...,I get my breakfast from Chicken Republic almos...,I get my breakfast from Chicken Republic almos...,2,chicken,positive,meal,11,I get my breakfast from Chicken Republic almos...,I get my breakfast from Chicken Republic almos...


In [ ]:
unique_reviews_list = list(set(reviews_list))

### Generating Summaries

In [ ]:
device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device=device,
    torch_dtype=torch.float16
)

prompts = []
for text in unique_reviews_list:
    messages = [
        {"role": "user", "content": f"Summarize the following customer reviews. Provide only a clean, concise paragraph summarizing the main points. Do not include introductory text, explanations, or formatting.\n\nReviews:\n\"{text}\""}
    ]
    # This automatically compiles the text into the exact prompt structure the model expects
    prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompts.append(prompt)

outputs = generator(
    prompts,
    max_new_tokens=150,
    do_sample=False,
    truncation=True,
    batch_size=2
)

# Extract only the freshly generated model response text
summary_series = [o[0]['generated_text'].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip() for o in outputs]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

In [ ]:
summary_df = pd.DataFrame({
    'concat_review':unique_reviews_list,
    'summary': summary_series})

In [ ]:
data2.columns

Index(['location', 'text', 'processed_review', 'simple_reviews', 'review_idx',
       'aspect', 'sentiment', 'Topic', 'Group_Label', 'concat_review_x',
       'concat_review_y'],
      dtype='object')

In [ ]:
absa_data = data2.merge(summary_df, on='concat_review', how='left')

KeyError: 'concat_review'

In [ ]:
absa_data.columns

In [ ]:
absa_data.head(3)

In [ ]:
absa_data.drop(['processed_review', 'review_idx', 'Group_Label', 'concat_review'], axis=1, inplace=True)

In [ ]:
absa_data.shape

In [ ]:
absa_data.head(3)

In [ ]:
absa_data.to_csv('chicken_republic_absa.csv', index=False)